# Task 1 — Population & Demographics Cleaning

## 0) Setup

In [13]:
import os
import pandas as pd

RAW = "Data_Raw_01"      # raw files folder (we only read from it)
OUT = "Data_Clean_02"    # output folder
os.makedirs(OUT, exist_ok=True)

# every message we print is also saved in this list, then written to cleaning_log.txt at the end
log_lines = []

def log(text=""):
    print(text)
    log_lines.append(str(text))

log("CLEANING LOG - Task 1: Population & Demographics")
log("=" * 60)

CLEANING LOG - Task 1: Population & Demographics


## 1) Files used

In [14]:
files = {
    "Population":           RAW + "/12.csv",
    "Age_0_14_Pct":         RAW + "/04.xls",
    "Age_65_Plus_Pct":      RAW + "/05.xls",
    "Urban_Population_Pct": RAW + "/06.xls",
}
dict_path = RAW + "/country_dictionary_v1.csv"

log("Files used:")
for name in files:
    log("  " + name + " <- " + files[name])
log("  Country Dictionary <- " + dict_path)

Files used:
  Population <- Data_Raw_01/12.csv
  Age_0_14_Pct <- Data_Raw_01/04.xls
  Age_65_Plus_Pct <- Data_Raw_01/05.xls
  Urban_Population_Pct <- Data_Raw_01/06.xls
  Country Dictionary <- Data_Raw_01/country_dictionary_v1.csv


## 2) Read the Country Dictionary (main reference for countries)

In [15]:
cd = pd.read_csv(dict_path, encoding="utf-8-sig")
cd = cd[["ISO3", "Country"]]

log("")
log("Country Dictionary: rows = " + str(len(cd)) + " | unique ISO3 = " + str(cd["ISO3"].nunique()))
cd.head()


Country Dictionary: rows = 196 | unique ISO3 = 196


,ISO3,Country
0,AFG,Afghanistan
1,AGO,Angola
2,ALB,Albania
3,AND,Andorra
4,ARE,United Arab Emirates


## 3) Read the 4 raw datasets (wide format)

In [16]:
raw = {}
log("")
log("--- RAW FILES ---")

for name in files:
    path = files[name]

    # World Bank files have a few title rows before the real header
    if path.endswith(".csv"):
        df = pd.read_csv(path, skiprows=4, encoding="utf-8-sig")
    else:
        df = pd.read_excel(path, sheet_name="Data", skiprows=3)

    rows_before = len(df)

    # drop empty extra columns (like 'Unnamed: 70')
    dropped = []
    for c in df.columns:
        if str(c).startswith("Unnamed") and df[c].isna().all():
            dropped.append(c)
    df = df.drop(columns=dropped)

    # the year columns are the columns whose name is a number (1960, 1961, ...)
    year_cols = []
    for c in df.columns:
        if str(c).isdigit():
            year_cols.append(c)

    log("")
    log("File: " + path + " -> indicator: " + name)
    log("  rows before cleaning = " + str(rows_before) + " | cols = " + str(df.shape[1]) + " | empty cols dropped = " + str(dropped))
    log("  year columns = " + str(len(year_cols)) + " (" + str(year_cols[0]) + "-" + str(year_cols[-1]) + ") | missing cells in year cols = " + str(int(df[year_cols].isna().sum().sum())))
    log("  Indicator: " + str(df["Indicator Name"].iloc[0]))

    raw[name] = df

raw["Population"].head()


--- RAW FILES ---

File: Data_Raw_01/12.csv -> indicator: Population
  rows before cleaning = 265 | cols = 70 | empty cols dropped = ['Unnamed: 70']
  year columns = 66 (1960-2025) | missing cells in year cols = 96
  Indicator: Population, total

File: Data_Raw_01/04.xls -> indicator: Age_0_14_Pct
  rows before cleaning = 265 | cols = 70 | empty cols dropped = []
  year columns = 66 (1960-2025) | missing cells in year cols = 66
  Indicator: Population ages 0-14 (% of total population)

File: Data_Raw_01/05.xls -> indicator: Age_65_Plus_Pct
  rows before cleaning = 265 | cols = 70 | empty cols dropped = []
  year columns = 66 (1960-2025) | missing cells in year cols = 66
  Indicator: Population ages 65 and above (% of total population)

File: Data_Raw_01/06.xls -> indicator: Urban_Population_Pct
  rows before cleaning = 265 | cols = 70 | empty cols dropped = []
  year columns = 66 (1960-2025) | missing cells in year cols = 66
  Indicator: Urban population (% of total population)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108727.0,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,623369401.0,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,34700612.0,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106384410.0,108754449.0,...,428797338.0,440150152.0,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,29183070.0,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0


## 4) Wide → Long

In [17]:
long = {}
log("")
log("--- WIDE -> LONG ---")

for name in raw:
    df = raw[name]

    year_cols = []
    for c in df.columns:
        if str(c).isdigit():
            year_cols.append(c)

    l = df.melt(id_vars=["Country Code"], value_vars=year_cols, var_name="Year", value_name="Value")
    l = l.rename(columns={"Country Code": "ISO3"})
    l["Year"] = l["Year"].astype(int)
    l["Value"] = pd.to_numeric(l["Value"], errors="coerce")   # missing stays NaN

    long[name] = l
    log(name + ": wide rows=" + str(len(df)) + " x years=" + str(len(year_cols)) + " -> long rows=" + str(len(l)) + " | years " + str(l["Year"].min()) + "-" + str(l["Year"].max()))

long["Population"].head()


--- WIDE -> LONG ---
Population: wide rows=265 x years=66 -> long rows=17490 | years 1960-2025
Age_0_14_Pct: wide rows=265 x years=66 -> long rows=17490 | years 1960-2025
Age_65_Plus_Pct: wide rows=265 x years=66 -> long rows=17490 | years 1960-2025
Urban_Population_Pct: wide rows=265 x years=66 -> long rows=17490 | years 1960-2025


,ISO3,Year,Value
0,ABW,1960,54922.0
1,AFE,1960,130075728.0
2,AFG,1960,9035043.0
3,AFW,1960,97630925.0
4,AGO,1960,5231654.0


## 5) Filter using the Country Dictionary (+ find aggregates)

In [18]:
# The World Bank metadata sheet tells us which codes are aggregates (they have no Region)
meta = pd.read_excel(files["Age_0_14_Pct"], sheet_name="Metadata - Countries")
aggregates = set(meta.loc[meta["Region"].isna(), "Country Code"])
all_meta_codes = set(meta["Country Code"])
log("")
log("World Bank metadata sheet: " + str(len(all_meta_codes)) + " codes | flagged as aggregates (Region empty) = " + str(len(aggregates)))

dict_codes = set(cd["ISO3"])
wb_names = raw["Population"].set_index("Country Code")["Country Name"]

filtered = {}
excluded_codes = set()
log("")
log("--- FILTER BY COUNTRY DICTIONARY ---")

for name in long:
    l = long[name]
    codes_before = set(l["ISO3"])
    excluded = codes_before - dict_codes
    excluded_codes = excluded_codes | excluded

    rows_before = len(l)
    l = l[l["ISO3"].isin(dict_codes)].copy()

    log("")
    log(name + ": codes before=" + str(len(codes_before)) + " | kept (in dictionary)=" + str(l["ISO3"].nunique()) + " | excluded=" + str(len(excluded)))
    log("  dictionary countries NOT found in this file: " + str(sorted(dict_codes - codes_before)))
    log("  long rows before filter=" + str(rows_before) + " -> after filter=" + str(len(l)) + " (removed " + str(rows_before - len(l)) + " rows)")

    filtered[name] = l

# group the excluded codes by reason
excluded_aggregates = []
excluded_not_in_dict = []
for code in sorted(excluded_codes):
    if code in aggregates:
        excluded_aggregates.append(code)
    else:
        excluded_not_in_dict.append(code)

log("")
log("Total distinct excluded codes = " + str(len(excluded_codes)))
log("  Aggregates (WB metadata: no Region): " + str(len(excluded_aggregates)) + " codes")
log("  Not aggregates and NOT in Country Dictionary: " + str(len(excluded_not_in_dict)) + " codes")

log("")
log("Excluded aggregates:")
for code in excluded_aggregates:
    log("     " + code + ": " + str(wb_names[code]))
log("Excluded non-aggregates (for review by team lead):")
for code in excluded_not_in_dict:
    log("     " + code + ": " + str(wb_names[code]))


World Bank metadata sheet: 264 codes | flagged as aggregates (Region empty) = 47

--- FILTER BY COUNTRY DICTIONARY ---

Population: codes before=265 | kept (in dictionary)=194 | excluded=71
  dictionary countries NOT found in this file: ['COK', 'NIU']
  long rows before filter=17490 -> after filter=12804 (removed 4686 rows)

Age_0_14_Pct: codes before=265 | kept (in dictionary)=194 | excluded=71
  dictionary countries NOT found in this file: ['COK', 'NIU']
  long rows before filter=17490 -> after filter=12804 (removed 4686 rows)

Age_65_Plus_Pct: codes before=265 | kept (in dictionary)=194 | excluded=71
  dictionary countries NOT found in this file: ['COK', 'NIU']
  long rows before filter=17490 -> after filter=12804 (removed 4686 rows)

Urban_Population_Pct: codes before=265 | kept (in dictionary)=194 | excluded=71
  dictionary countries NOT found in this file: ['COK', 'NIU']
  long rows before filter=17490 -> after filter=12804 (removed 4686 rows)

Total distinct excluded codes = 71

## 6) Validation for each table (before merge)

In [19]:
log("")
log("--- PER-TABLE VALIDATION ---")

for name in filtered:
    l = filtered[name]
    dup = l.duplicated(["ISO3", "Year"]).sum()
    log(name + ": rows=" + str(len(l)) + " | countries=" + str(l["ISO3"].nunique()) + " | years " + str(l["Year"].min()) + "-" + str(l["Year"].max()) + " | missing=" + str(l["Value"].isna().sum()) + " | dup(ISO3+Year)=" + str(dup))

pop_values = filtered["Population"]["Value"].dropna()
log("")
log("Population <= 0 values: " + str((pop_values <= 0).sum()))


--- PER-TABLE VALIDATION ---
Population: rows=12804 | countries=194 | years 1960-2025 | missing=30 | dup(ISO3+Year)=0
Age_0_14_Pct: rows=12804 | countries=194 | years 1960-2025 | missing=0 | dup(ISO3+Year)=0
Age_65_Plus_Pct: rows=12804 | countries=194 | years 1960-2025 | missing=0 | dup(ISO3+Year)=0
Urban_Population_Pct: rows=12804 | countries=194 | years 1960-2025 | missing=0 | dup(ISO3+Year)=0

Population <= 0 values: 0


## 7) Merge using ISO3 + Year only

In [20]:
keys = ["ISO3", "Year"]

m = filtered["Population"].rename(columns={"Value": "Population"})
log("")
log("--- MERGES ---")
log("Start (Population): rows=" + str(len(m)))

for name in ["Age_0_14_Pct", "Age_65_Plus_Pct", "Urban_Population_Pct"]:
    right = filtered[name].rename(columns={"Value": name})
    rows_before = len(m)
    m = m.merge(right, on=keys, how="left")
    log("Merge " + name + ": rows before=" + str(rows_before) + " | right rows=" + str(len(right)) + " | after=" + str(len(m)))
    if len(m) != rows_before:
        print("WARNING: row count changed after merge!")

# country name comes from the Dictionary (merge on ISO3 only)
rows_before = len(m)
m = m.merge(cd, on="ISO3", how="left")
log("Merge Country (from dictionary, on ISO3 only): rows before=" + str(rows_before) + " | after=" + str(len(m)))


--- MERGES ---
Start (Population): rows=12804
Merge Age_0_14_Pct: rows before=12804 | right rows=12804 | after=12804
Merge Age_65_Plus_Pct: rows before=12804 | right rows=12804 | after=12804
Merge Urban_Population_Pct: rows before=12804 | right rows=12804 | after=12804
Merge Country (from dictionary, on ISO3 only): rows before=12804 | after=12804


## 8) Final cleaning & data types

In [21]:
final_cols = ["ISO3", "Country", "Year", "Population", "Age_0_14_Pct", "Age_65_Plus_Pct", "Urban_Population_Pct"]
df = m[final_cols].copy()

df["Year"] = df["Year"].astype(int)
df["Population"] = df["Population"].astype("Int64")     # whole numbers, missing stays <NA>
df = df.sort_values(["ISO3", "Year"]).reset_index(drop=True)

dups = df.duplicated(["ISO3", "Year"]).sum()

log("")
log("FINAL: rows=" + str(len(df)) + " | countries=" + str(df["ISO3"].nunique()) + " | years " + str(df["Year"].min()) + "-" + str(df["Year"].max()))
log("Duplicates on ISO3+Year = " + str(dups))
log("Missing values per column:")
for c in df.columns:
    log("  " + c + ": " + str(df[c].isna().sum()))
log("dtypes: " + str(df.dtypes.astype(str).to_dict()))

# which countries have missing Population
missing_pop = df[df["Population"].isna()].groupby("ISO3")["Year"].agg(["min", "max", "count"])
log("Countries with missing Population by year range:")
log(missing_pop.to_string())

not_in_data = sorted(dict_codes - set(df["ISO3"]))
log("Countries in dictionary with no row in the data: " + str(not_in_data))

df.head()


FINAL: rows=12804 | countries=194 | years 1960-2025
Duplicates on ISO3+Year = 0
Missing values per column:
  ISO3: 0
  Country: 0
  Year: 0
  Population: 30
  Age_0_14_Pct: 0
  Age_65_Plus_Pct: 0
  Urban_Population_Pct: 0
dtypes: {'ISO3': 'object', 'Country': 'object', 'Year': 'int64', 'Population': 'Int64', 'Age_0_14_Pct': 'float64', 'Age_65_Plus_Pct': 'float64', 'Urban_Population_Pct': 'float64'}
Countries with missing Population by year range:
       min   max  count
ISO3                   
PSE   1960  1989     30
Countries in dictionary with no row in the data: ['COK', 'NIU']


,ISO3,Country,Year,Population,Age_0_14_Pct,Age_65_Plus_Pct,Urban_Population_Pct
0,AFG,Afghanistan,1960,9035043,41.946754,2.823362,8.122551
1,AFG,Afghanistan,1961,9214083,42.028588,2.809357,8.389595
2,AFG,Afghanistan,1962,9404406,42.112814,2.792835,8.662664
3,AFG,Afghanistan,1963,9604487,42.235952,2.775057,8.942709
4,AFG,Afghanistan,1964,9814318,42.409539,2.757792,9.231134


## 9) Save the clean CSV

In [22]:
csv_path = OUT + "/demographics_clean.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

check = pd.read_csv(csv_path, encoding="utf-8-sig")
log("")
log("Saved " + csv_path + ": rows=" + str(len(check)) + " | cols=" + str(list(check.columns)))


Saved Data_Clean_02/demographics_clean.csv: rows=12804 | cols=['ISO3', 'Country', 'Year', 'Population', 'Age_0_14_Pct', 'Age_65_Plus_Pct', 'Urban_Population_Pct']


## 10) Final checklist

In [23]:
checks = {}
checks["Columns exactly as required"] = list(check.columns) == final_cols
checks["Duplicates on ISO3+Year = 0"] = check.duplicated(["ISO3", "Year"]).sum() == 0
checks["Year is integer"] = pd.api.types.is_integer_dtype(check["Year"])
checks["Population numeric and > 0 where present"] = pd.api.types.is_numeric_dtype(check["Population"]) and (check["Population"].dropna() > 0).all()
checks["% columns numeric"] = (pd.api.types.is_numeric_dtype(check["Age_0_14_Pct"])
                               and pd.api.types.is_numeric_dtype(check["Age_65_Plus_Pct"])
                               and pd.api.types.is_numeric_dtype(check["Urban_Population_Pct"]))
checks["All ISO3 exist in Country Dictionary"] = set(check["ISO3"]).issubset(dict_codes)
checks["No aggregates left (WB metadata)"] = len(set(check["ISO3"]) & aggregates) == 0
checks["No zero Population"] = (check["Population"] == 0).sum() == 0
checks["Row count = countries x years"] = len(check) == check["ISO3"].nunique() * check["Year"].nunique()

log("")
log("--- FINAL CHECKLIST ---")
for k in checks:
    if checks[k]:
        log("  [PASS] " + k)
    else:
        log("  [FAIL] " + k)


--- FINAL CHECKLIST ---
  [PASS] Columns exactly as required
  [PASS] Duplicates on ISO3+Year = 0
  [PASS] Year is integer
  [PASS] Population numeric and > 0 where present
  [PASS] % columns numeric
  [PASS] All ISO3 exist in Country Dictionary
  [PASS] No aggregates left (WB metadata)
  [PASS] No zero Population
  [PASS] Row count = countries x years


## 11) Write the cleaning log

In [24]:
log("")
log("--- SUMMARY ---")
log("Rows per dataset: raw (wide) = " + str(len(raw["Population"])) + " each | after Wide->Long = " + str(len(long["Population"])) + " | after dictionary filter = " + str(len(filtered["Population"])) + " | final merged = " + str(len(check)))
log("Renamed columns: 'Country Code' -> ISO3 | Year columns (1960..) melted into Year | 'Value' -> Population / Age_0_14_Pct / Age_65_Plus_Pct / Urban_Population_Pct | Country name taken from Dictionary 'Country'")
log("New columns calculated: none")
log("")
log("ISSUES / NOTES (not fixed manually - to be sent to team lead for central Dictionary update):")
log("  1) In Dictionary but NOT in World Bank files -> no rows in output: " + str(not_in_data))
log("  2) World Bank codes NOT in Dictionary and not aggregates -> excluded (" + str(len(excluded_not_in_dict)) + "): " + ", ".join(excluded_not_in_dict))
log("  3) Missing Population in raw file (left as NaN, no filling):")
for iso in missing_pop.index:
    log("       " + iso + ": " + str(int(missing_pop.loc[iso, "min"])) + "-" + str(int(missing_pop.loc[iso, "max"])) + " (" + str(int(missing_pop.loc[iso, "count"])) + " rows)")
log("  4) 12.csv has an empty trailing column 'Unnamed: 70' -> dropped (no data lost).")
log("Raw files modified: none (read only)")
log("fillna(0): not used | interpolation: not used | Missing values left as NaN")
log("Rows manually deleted: none. Only rows whose code is not in the Country Dictionary were excluded.")

with open(OUT + "/cleaning_log.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))
print("saved: demographics_clean.csv + cleaning_log.txt in", OUT)


--- SUMMARY ---
Rows per dataset: raw (wide) = 265 each | after Wide->Long = 17490 | after dictionary filter = 12804 | final merged = 12804
Renamed columns: 'Country Code' -> ISO3 | Year columns (1960..) melted into Year | 'Value' -> Population / Age_0_14_Pct / Age_65_Plus_Pct / Urban_Population_Pct | Country name taken from Dictionary 'Country'
New columns calculated: none

ISSUES / NOTES (not fixed manually - to be sent to team lead for central Dictionary update):
  1) In Dictionary but NOT in World Bank files -> no rows in output: ['COK', 'NIU']
  2) World Bank codes NOT in Dictionary and not aggregates -> excluded (24): ABW, ASM, BMU, CHI, CUW, CYM, FRO, GIB, GRL, GUM, HKG, IMN, INX, MAC, MAF, MNP, NCL, PRI, PYF, SXM, TCA, VGB, VIR, XKX
  3) Missing Population in raw file (left as NaN, no filling):
       PSE: 1960-1989 (30 rows)
  4) 12.csv has an empty trailing column 'Unnamed: 70' -> dropped (no data lost).
Raw files modified: none (read only)
fillna(0): not used | interpolatio